# CSE 438 — Digital Image Processing · Assignment **Part B**
## Self-Supervised Learning for Semantic Segmentation — Data Preparation (Task A)

**Group:** 01 · Department of Computer Science and Engineering, East West University

| # | Name | Student ID |
|---|------|-----------|
| 1 | Md. Asif Hossain | 2022-3-60-007 |
| 2 | Nabil Subhan | 2022-3-60-063 |
| 3 | K M Nudar | 2022-3-60-234 |

**Dataset:** LiTS 256×256 (liver + tumor CT) — *same as Part A*.

This notebook performs **Task A**: it loads the **identical leakage-safe split produced in Part A (NB0)**
*unmodified*, remaps each split to its self-supervised role, builds the three role loaders, and produces
the required sanity visualisation. No re-splitting, no re-shuffling.

| Part-A split | New Part-B role | Masks |
|---|---|---|
| **train** | **unlabelled** SSL pretraining corpus (Task B) | discarded — images only |
| **validation** | **labelled** fine-tuning set (Task D) | used (image + mask) |
| **test** | downstream **monitoring + final evaluation** (Tasks D & E) | used at eval — *double role* |

> **Test-split double role (report §6.3):** the test split is used both to select the fine-tuning checkpoint
> *and* to report final metrics. This is a deliberate simplification given the small labelled set; it makes the
> reported numbers **mildly optimistic** versus a truly-held-out test set. Disclosed here and in the report.

**Inputs required (attach on Kaggle):** (1) the Part-A NB0 output dataset containing `split.json`, and
(2) the original LiTS 256×256 image/mask dataset.

In [ ]:
# =====================================================================
# Setup, version pins, seeding, and loading of the Part-A split (UNMODIFIED)
# =====================================================================
import os, re, json, hashlib, random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch

print("numpy", np.__version__, "| torch", torch.__version__, "| albumentations", A.__version__)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CLASS_NAMES = ["background", "liver", "tumor"]; NUM_CLASSES = 3
SEG_SIZE = 256      # labelled fine-tune + test resolution (Part-A parity)
SSL_SIZE = 224      # SSL pretraining view resolution (ImageNet-checkpoint & ViT-patch friendly: 224=16x14=14x16)
IMAGENET_MEAN = (0.485, 0.456, 0.406); IMAGENET_STD = (0.229, 0.224, 0.225)
WORK = Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)

# ---- locate the Part-A split artifact (attached as a Kaggle dataset input) ----
split_paths = list(Path("/kaggle/input").rglob("split.json"))
assert split_paths, "Attach the Part-A NB0 output dataset (must contain split.json)."
SPLIT_JSON = split_paths[0]
split_meta = json.load(open(SPLIT_JSON))
split = split_meta["slices"]                       # {'train':[...], 'val':[...], 'test':[...]} -- UNTOUCHED
print("loaded Part-A split :", SPLIT_JSON)
print("label rule          :", split_meta["label_rule"])
print("seed / ratios       :", split_meta["seed"], split_meta["ratios"])

# ---- locate the raw LiTS image/mask folders (re-detect, fall back to split.json dirs) ----
imgdir_saved = Path(split_meta["dirs"]["images"])
if imgdir_saved.exists():
    IMG_DIR, LIVER_DIR, TUMOR_DIR = imgdir_saved, Path(split_meta["dirs"]["liver"]), Path(split_meta["dirs"]["tumor"])
else:
    cands = list(Path("/kaggle/input").rglob("Thesis_data"))
    ROOT = cands[0] if cands else Path("/kaggle/input")
    IMG_DIR   = next(p for p in ROOT.iterdir() if p.is_dir() and "image" in p.name.lower())
    LIVER_DIR = next(p for p in ROOT.iterdir() if p.is_dir() and "liver" in p.name.lower())
    TUMOR_DIR = next(p for p in ROOT.iterdir() if p.is_dir() and "tumor" in p.name.lower())
print("image dir           :", IMG_DIR)

def vskey(p):
    n = [int(x) for x in re.findall(r"\d+", Path(p).stem)]
    return (n[0], n[1]) if len(n) >= 2 else (n[0], -1)
img_by = {vskey(p): p for p in IMG_DIR.glob("*.png")}
liv_by = {vskey(p): p for p in LIVER_DIR.glob("*.png")}
tum_by = {vskey(p): p for p in TUMOR_DIR.glob("*.png")}
print("paired images       :", len(img_by))

## Task A.1 — Split role mapping & leakage verification

We do **not** touch the split: the same `train / val / test` file lists from Part A are simply re-labelled with
their Part-B roles. To *prove* nothing was re-shuffled we (a) assert the three **volume** sets stay disjoint
(the leakage-safe guarantee) and (b) print a short **SHA-256 fingerprint** of each role's sorted slice list as a
provenance stamp that other notebooks can re-check.

In [ ]:
# =====================================================================
# Task A.1 - role mapping + leakage verification + exact counts
# =====================================================================
def vset(lst):        return set(int(x.split("_")[0]) for x in lst)
def fingerprint(lst): return hashlib.sha256("\n".join(sorted(lst)).encode()).hexdigest()[:12]

# leakage guard: the three VOLUME sets must be disjoint (no re-shuffle across roles)
va, vb, vc = vset(split["train"]), vset(split["val"]), vset(split["test"])
assert not (va & vb) and not (va & vc) and not (vb & vc), "LEAKAGE: volumes overlap across roles!"

rows = [("train", "unlabelled_pretrain", split["train"]),
        ("val",   "labelled_finetune",   split["val"]),
        ("test",  "eval_monitor_test",   split["test"])]
print(f"{'Part-A':<7}{'Part-B role':<22}{'#slices':>8}{'#vols':>7}   fingerprint")
for pa, role, sl in rows:
    print(f"{pa:<7}{role:<22}{len(sl):>8}{len(vset(sl)):>7}   {fingerprint(sl)}")

n_lab, n_pre = len(split["val"]), len(split["train"])
print("\nLEAKAGE CHECK PASSED - roles are volume-disjoint (Part-A split reused unmodified).")
print(f"label-efficiency ratio : fine-tune on {n_lab} labelled slices vs {n_pre} used by the "
      f"Part-A supervised baseline = {n_lab/n_pre:.3f}  (~{round(100*n_lab/n_pre)}% of the labels)")

In [ ]:
# =====================================================================
# Task A.1 - save role artifacts for the 4 method notebooks + final comparison
# =====================================================================
import shutil
roles = {"unlabelled_pretrain": split["train"],
         "labelled_finetune":   split["val"],
         "eval_monitor_test":   split["test"]}
roles_out = {
    "seed": SEED, "num_classes": NUM_CLASSES, "class_names": CLASS_NAMES,
    "label_rule": split_meta["label_rule"], "dirs": split_meta["dirs"],
    "seg_size": SEG_SIZE, "ssl_size": SSL_SIZE,
    "roles": roles,
    "counts": {k: len(v) for k, v in roles.items()},
    "fingerprints": {pa: fingerprint(sl) for pa, _, sl in
                     [("train", 0, split["train"]), ("val", 0, split["val"]), ("test", 0, split["test"])]},
    "source_split_json": str(SPLIT_JSON),
}
json.dump(roles_out, open(WORK/"partB_split_roles.json", "w"), indent=2)
shutil.copy(SPLIT_JSON, WORK/"split.json")                              # carry the ORIGINAL forward, unchanged
cw = Path(SPLIT_JSON).parent/"class_weights.json"
if cw.exists(): shutil.copy(cw, WORK/"class_weights.json")
print("saved:", WORK/"partB_split_roles.json")
print("copied split.json unchanged -> the 4 method notebooks attach THIS notebook's output and read it.")

## Task A.2 — The three role loaders

* **Train → unlabelled:** yields **augmented image views only, no masks** (the SSL pretext task supplies its own
  target). The exact view pipeline **differs per method** — all four are defined below.
* **Validation → labelled:** reuses the **Part-A synchronized image+mask** augmentation — this small labelled set
  is the fine-tuning data.
* **Test → eval:** **deterministic** resize + normalize (no augmentation) — used for both checkpoint monitoring and
  final metrics.

**Input format:** we reuse Part A's **2.5D** input (3 channels = adjacent slices `[i-1, i, i+1]`), which maps
directly onto the 3 RGB channels the ImageNet-pretrained SSL encoders expect.

In [ ]:
# =====================================================================
# Task A.2 - shared image loaders (2.5D input + label fusion, reused from Part A)
# =====================================================================
def load_25d(vol, sl):                          # 3 channels = adjacent slices [i-1, i, i+1]
    center = np.array(Image.open(img_by[(vol, sl)]).convert("L"))
    prev = img_by.get((vol, sl-1)); nxt = img_by.get((vol, sl+1))
    p = np.array(Image.open(prev).convert("L")) if prev is not None else center
    n = np.array(Image.open(nxt).convert("L")) if nxt is not None else center
    return np.stack([p, center, n], axis=-1)    # HxWx3 uint8

def fuse_label(vol, sl):                         # bg=0, liver=1, tumor=2 (tumor precedence)
    liv = np.array(Image.open(liv_by[(vol, sl)]).convert("L")) > 0
    tum = np.array(Image.open(tum_by[(vol, sl)]).convert("L")) > 0
    lab = np.zeros(liv.shape, np.uint8); lab[liv] = 1; lab[tum] = 2
    return lab

def parse_ids(slice_ids):
    return [tuple(int(x) for x in s.split("_")) for s in slice_ids]

In [ ]:
# =====================================================================
# Task A.2 - per-method augmentation pipelines + role datasets
# =====================================================================
# ---- SSL pretraining VIEWS (image-only, output SSL_SIZE). Color aug softened for 3-slice CT stacks. ----
# SimCLR / BYOL : strong crop + photometric (two views drive the contrastive / prediction objective)
ssl_view = A.Compose([
    A.RandomResizedCrop(size=(SSL_SIZE, SSL_SIZE), scale=(0.2, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(0.2, 0.2, 0.2, 0.05, p=0.6),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])
# MAE : masking IS the pretext -> minimal aug
mae_view = A.Compose([
    A.RandomResizedCrop(size=(SSL_SIZE, SSL_SIZE), scale=(0.4, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])
# DINOv2 : multi-crop (2 global + N local)
dino_global = A.Compose([
    A.RandomResizedCrop(size=(SSL_SIZE, SSL_SIZE), scale=(0.4, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])
dino_local = A.Compose([
    A.RandomResizedCrop(size=(96, 96), scale=(0.05, 0.4)),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])
METHOD_VIEWS = {"simclr": ssl_view, "byol": ssl_view, "mae": mae_view,
                "dinov2": {"global": dino_global, "local": dino_local}}

# ---- labelled fine-tune (val) + eval (test): Part-A synchronized image+mask transforms ----
label_tf = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.Affine(scale=(0.9, 1.1), translate_percent=0.06, rotate=(-15, 15), p=0.5),
    A.OneOf([A.ElasticTransform(alpha=40, sigma=6),
             A.GridDistortion(num_steps=5, distort_limit=0.2)], p=0.25),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.3), A.GaussNoise(p=0.2),
    A.Resize(SEG_SIZE, SEG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])
eval_tf = A.Compose([
    A.Resize(SEG_SIZE, SEG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])

# ---- role datasets ----
class SSLViewDataset(torch.utils.data.Dataset):        # TRAIN: image VIEWS ONLY (no mask)
    def __init__(self, slice_ids, view, n_views=2):
        self.ids = parse_ids(slice_ids); self.view = view; self.n = n_views
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        img = load_25d(*self.ids[i])
        return [self.view(image=img)["image"] for _ in range(self.n)]

class DinoMultiCropDataset(torch.utils.data.Dataset):  # TRAIN: 2 global + n_local local crops
    def __init__(self, slice_ids, n_local=6):
        self.ids = parse_ids(slice_ids); self.n_local = n_local
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        img = load_25d(*self.ids[i])
        return ([dino_global(image=img)["image"] for _ in range(2)] +
                [dino_local(image=img)["image"] for _ in range(self.n_local)])

class LabelledDataset(torch.utils.data.Dataset):       # VAL / TEST: image + fused mask
    def __init__(self, slice_ids, transform):
        self.ids = parse_ids(slice_ids); self.tf = transform
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        vol, sl = self.ids[i]
        out = self.tf(image=load_25d(vol, sl), mask=fuse_label(vol, sl))
        return out["image"], out["mask"].long()

print("pipelines ready:", list(METHOD_VIEWS.keys()), "| label_tf + eval_tf @", SEG_SIZE)

In [ ]:
# =====================================================================
# Task A.2 - instantiate the three role loaders (SimCLR 2-view shown for the pretrain role)
# =====================================================================
from torch.utils.data import DataLoader
pretrain_ds = SSLViewDataset(roles["unlabelled_pretrain"], ssl_view, n_views=2)
finetune_ds = LabelledDataset(roles["labelled_finetune"], label_tf)
test_ds     = LabelledDataset(roles["eval_monitor_test"], eval_tf)
print("pretrain (unlabelled):", len(pretrain_ds),
      "| finetune (labelled):", len(finetune_ds), "| test (eval):", len(test_ds))

views = pretrain_ds[0]
print("SSL sample  -> ", len(views), "views, each", tuple(views[0].shape))     # (3, SSL_SIZE, SSL_SIZE)
xb, yb = finetune_ds[0]
print("labelled sample -> image", tuple(xb.shape), "| mask", tuple(yb.shape),
      "| classes", torch.unique(yb).tolist())
xt, yt = test_ds[0]
print("test sample     -> image", tuple(xt.shape), "| mask", tuple(yt.shape))

## Task A.3 — Sanity check

Two things must be visually confirmed **before any pretraining**:
1. the SSL pipeline produces **two different augmented views of the *same* train image** (the contrastive signal), and
2. the labelled pipeline keeps the **validation image and its mask aligned** after synchronized augmentation.

In [ ]:
# =====================================================================
# Task A.3 - sanity grid: (cols 1-2) two SSL views of one train image ; (col 3) val image + mask overlay
# =====================================================================
def denorm(t):
    x = t.permute(1, 2, 0).numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(x, 0, 1)
cmap = plt.matplotlib.colors.ListedColormap(["black", "gold", "red"])

# prefer tumor-containing val slices for the overlay column so the mask is visible
man_paths = list(Path("/kaggle/input").rglob("manifest.csv"))
val_show = roles["labelled_finetune"][:3]
if man_paths:
    mf = pd.read_csv(man_paths[0]); mf["key"] = mf.volume.astype(str)+"_"+mf.slice.astype(str)
    tset = set(mf[mf.has_tumor == 1].key)
    val_show = ([s for s in roles["labelled_finetune"] if s in tset][:3] + val_show)[:3]

fig, ax = plt.subplots(3, 3, figsize=(10, 10))
for r in range(3):
    v1, v2 = SSLViewDataset(roles["unlabelled_pretrain"], ssl_view, 2)[r]
    ax[r, 0].imshow(denorm(v1)[..., 1], cmap="gray"); ax[r, 0].set_title(f"train {r}: SSL view 1", fontsize=8)
    ax[r, 1].imshow(denorm(v2)[..., 1], cmap="gray"); ax[r, 1].set_title("SSL view 2 (same image)", fontsize=8)
    xb, yb = LabelledDataset([val_show[r]], eval_tf)[0]; im = denorm(xb)[..., 1]; m = yb.numpy()
    ax[r, 2].imshow(im, cmap="gray"); ax[r, 2].imshow(m, cmap=cmap, vmin=0, vmax=2, alpha=0.45)
    ax[r, 2].set_title(f"val {val_show[r]} + mask", fontsize=8)
    for c in range(3): ax[r, c].axis("off")
plt.suptitle("Task A sanity — cols 1-2: two augmented views share ONE train image (no mask);  "
             "col 3: val image+mask stay aligned", y=1.01, fontsize=10)
plt.tight_layout(); plt.savefig(WORK/"partB_sanity.png", dpi=120); plt.show()

## Summary — what the method notebooks consume

This notebook's output (`partB_split_roles.json` + the carried-forward `split.json`) is published as a Kaggle
dataset and attached to **every** method notebook (SimCLR, BYOL, MAE, DINOv2) and the final comparison. Each of
those notebooks reads the **same** role lists — guaranteeing the Part-A-vs-Part-B comparison is apples-to-apples.

**Reused identically across all four methods:**
* `roles["unlabelled_pretrain"]` (train, images only) → 50-epoch SSL pretext task (Task B)
* `roles["labelled_finetune"]` (val, image+mask) → 50-epoch supervised fine-tuning (Task D)
* `roles["eval_monitor_test"]` (test) → monitoring + final metrics (Tasks D & E)
* `load_25d`, `fuse_label`, `label_tf`, `eval_tf`, and the per-method view pipelines above.

Next notebook: **`lits-simclr`** — attaches the **Part-A DeepLabV3 ASPP decoder** to a SimCLR-pretrained ResNet-50.